# Epsilon Curves — Differential Privacy Benchmark Analysis

This notebook analyses the differential privacy benchmark results collected in T52.2.
It reads committed JSON artifacts from `demos/results/` — no live training is performed.

**Benchmark grid:** noise_multiplier ∈ [1.0, 5.0, 10.0] × epochs ∈ [50, 100] × sample_size = 1000

**Datasets:**
- `customers` — 1 000 rows, 8 columns (id, first_name, last_name, email, ssn, phone, address, created_at)
- `orders` — 1 000 rows synthesised, 5 columns (id, customer_id, order_date, total_amount, status)

Jump to: [Methodology](#methodology) | [Epsilon vs Noise](#epsilon-vs-noise-multiplier) | [Fidelity](#epsilon-vs-statistical-fidelity) | [Schema Complexity](#epsilon-vs-schema-complexity) | [Correlations](#correlation-preservation) | [FK Integrity](#fk-integrity-verification) | [Limitations](#honest-limitations)


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless backend — safe for nbconvert and CI
import matplotlib.pyplot as plt
import numpy as np

# Resolve results and figures directories relative to this notebook
_DEMOS_DIR = Path("demos")
_RESULTS_DIR = _DEMOS_DIR / "results"
_FIGURES_DIR = _DEMOS_DIR / "figures"

print(f"Results: {_RESULTS_DIR}")
print(f"Figures: {_FIGURES_DIR}")

In [ ]:
def load_results(path: Path) -> list[dict]:
    """Load result rows from a committed JSON artifact."""
    return json.loads(path.read_text(encoding="utf-8"))["rows"]


customers_rows = load_results(_RESULTS_DIR / "benchmark_customers_v1.json")
orders_rows = load_results(_RESULTS_DIR / "benchmark_orders_v1.json")

completed_c = [r for r in customers_rows if r["status"] == "COMPLETED"]
completed_o = [r for r in orders_rows if r["status"] == "COMPLETED"]
failed_c = [r for r in customers_rows if r["status"] == "FAILED"]

print(f"Customers: {len(completed_c)} COMPLETED, {len(failed_c)} FAILED")
print(f"Orders: {len(completed_o)} COMPLETED")

## Methodology

### Hardware

All benchmarks were run on the same machine:

| Field | Value |
|-------|-------|
| CPU | ARM (Apple Silicon) |
| CPU cores | 10 |
| RAM | 24 GB |
| GPU | None (CPU-only run) |
| OS | Darwin 24.5.0 |

GPU was not available for these runs. cuDNN non-determinism is therefore not
a concern for this dataset, but results may differ on GPU-equipped machines.

### Software Versions

| Component | Version |
|-----------|---------|
| Python | 3.14.1 |
| SDV (CTGANSynthesizer) | >= 1.17.0, < 2.0.0 |
| Opacus (DP-SGD / RDP accountant) | >= 1.4.0, < 2.0.0 |
| Torch | >= 2.5.0, < 3.0.0 |

### Differential Privacy Accountant

Opacus uses the **Renyi Differential Privacy (RDP)** accountant.
Privacy budget parameters:
- **delta (delta):** 1e-5 (1 in 100 000)
- **seed:** 42 (fixed for reproducibility; production uses cryptographically random seed)

### Dataset Descriptions

| Dataset | Sample size | Columns | Column types |
|---------|-------------|---------|-----|
| customers | 1 000 rows | 8 | id (int), first_name (str), last_name (str), email (str), ssn (str), phone (str), address (str), created_at (datetime) |
| orders | 1 000 rows | 5 | id (int), customer_id (int FK), order_date (datetime), total_amount (float), status (enum) |

### Parameter Grid

The reduced benchmark grid contains 6 cells per dataset (12 total):

| noise_multiplier | epochs | sample_size |
|-----------------|--------|-------------|
| 1.0 | 50 | 1 000 |
| 1.0 | 100 | 1 000 |
| 5.0 | 50 | 1 000 |
| 5.0 | 100 | 1 000 |
| 10.0 | 50 | 1 000 |
| 10.0 | 100 | 1 000 |

The originally planned grid was larger (3 sigma x 3 epoch counts x 5 sample sizes
x 3 seeds = 135 cells per dataset). The reduced grid was chosen for practical
execution time on CPU-only hardware.

### Wall-Time Scope

wall_time_seconds measures elapsed real time from synthesiser call entry to result
recording. It includes: CTGAN model fit, Opacus DP-SGD accounting, and synthetic
data generation. It does NOT include data loading, schema reflection, or harness overhead.

### Limitations of Reduced Grid

The `sample_size` dimension was dropped entirely from the executed grid.
All runs used sample_size=1000. Any claim about how epsilon scales with
dataset size cannot be supported by this benchmark. See the
[Honest Limitations](#honest-limitations) section for full details.


## Epsilon vs. Noise Multiplier

The central relationship in differential privacy: higher noise (sigma) -> lower epsilon
(stronger privacy). The plot shows measured epsilon for all completed runs.

**Key observations:**
- The inverse relationship holds: sigma=10 achieves epsilon ~1.2-3.0; sigma=1 achieves epsilon ~25-40
- The failed run (customers, sigma=1.0, epochs=100) is shown as a red X
- Orders achieves lower epsilon than customers at same sigma/epochs settings


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for rows, label, color, marker in [
    (customers_rows, "Customers (8 cols)", "#2563eb", "o"),
    (orders_rows, "Orders (5 cols)", "#16a34a", "s"),
]:
    completed = [r for r in rows if r["status"] == "COMPLETED"]
    xs = [r["noise_multiplier"] for r in completed]
    ys = [r["actual_epsilon"] for r in completed]
    ax.scatter(xs, ys, label=label, color=color, marker=marker, s=80, zorder=3)
    for x, y, r in zip(xs, ys, completed, strict=False):
        ax.annotate(
            f"ep={r['epochs']}",
            (x, y),
            xytext=(6, 2),
            textcoords="offset points",
            fontsize=7,
            color=color,
        )

for r in failed_c:
    ax.scatter(
        r["noise_multiplier"],
        0.5,
        marker="x",
        color="#dc2626",
        s=120,
        linewidths=2,
        zorder=4,
        label=f"FAILED: {r.get('error_type')} (ep={r['epochs']})",
    )

ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Measured epsilon")
ax.set_title("Epsilon vs. Noise Multiplier\n(lower sigma -> higher epsilon; X = budget exhausted)")
ax.legend(loc="upper right", fontsize=8)
ax.grid(visible=True, alpha=0.3)
ax.set_xscale("log")
ax.set_yscale("log")
plt.tight_layout()
plt.show()

## Epsilon vs. Statistical Fidelity

Statistical fidelity measures how well the synthesised distribution matches the real distribution.

- **Categorical columns**: chi2 p-value (higher = synthesised dist. matches real)
- **Numeric columns**: KS statistic (lower = more similar distributions)

**Key observations:**
- Most categorical columns achieve p > 0.05 at all epsilon levels
- `created_at` achieves p ~4.4e-103 — CTGAN weakness with datetime columns
- No clear fidelity improvement with lower epsilon at this grid scale


In [ ]:
def mean_chi2_pvalue(row: dict) -> float | None:
    col_metrics = row.get("column_metrics")
    if not col_metrics:
        return None
    pvalues = [
        v["chi2_pvalue"] for v in col_metrics.values() if isinstance(v, dict) and "chi2_pvalue" in v
    ]
    return sum(pvalues) / len(pvalues) if pvalues else None


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, rows, title, color in [
    (axes[0], customers_rows, "Customers (8 cols)", "#2563eb"),
    (axes[1], orders_rows, "Orders (5 cols)", "#16a34a"),
]:
    completed = [r for r in rows if r["status"] == "COMPLETED"]
    data = [
        (r["actual_epsilon"], mean_chi2_pvalue(r))
        for r in completed
        if r.get("actual_epsilon") is not None and mean_chi2_pvalue(r) is not None
    ]
    if data:
        xs, ys = zip(*data, strict=False)
        ax.scatter(list(xs), list(ys), color=color, s=80, zorder=3)
    ax.set_xlabel("Measured epsilon")
    ax.set_ylabel("Mean chi2 p-value (categorical cols)")
    ax.set_title(title)
    ax.set_ylim(0, 1.05)
    ax.axhline(0.05, linestyle="--", color="gray", alpha=0.5, label="p=0.05 threshold")
    ax.legend(fontsize=8)
    ax.grid(visible=True, alpha=0.3)

fig.suptitle("Epsilon vs. Statistical Fidelity", fontsize=12)
fig.tight_layout()
plt.show()

## Epsilon vs. Schema Complexity

> **Limitation:** Our benchmark grid has only `sample_size=1000` for all runs.
> We cannot plot 'epsilon vs. dataset size' because there is no size variation.
> See the [Honest Limitations](#honest-limitations) section.

Instead, we use schema complexity (number of columns) as a proxy:
- customers = 8 columns
- orders = 5 columns

Both schemas use sample_size=1000. Any epsilon difference reflects schema complexity,
not dataset size. Orders consistently achieves lower epsilon, consistent with
fewer columns reducing per-step sensitivity.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

schema_data = {
    "Customers\n(8 cols)": {"rows": customers_rows, "color": "#2563eb"},
    "Orders\n(5 cols)": {"rows": orders_rows, "color": "#16a34a"},
}

x_positions = np.arange(len(schema_data))
width = 0.35

for i, (label, info) in enumerate(schema_data.items()):
    completed = [r for r in info["rows"] if r["status"] == "COMPLETED"]
    epsilons = [r["actual_epsilon"] for r in completed]
    if epsilons:
        mean_eps = np.mean(epsilons)
        std_eps = np.std(epsilons)
        ax.bar(
            x_positions[i],
            mean_eps,
            width,
            yerr=std_eps,
            label=label,
            color=info["color"],
            alpha=0.8,
            capsize=6,
        )
        for eps in epsilons:
            ax.scatter(x_positions[i], eps, color="black", s=20, zorder=5, alpha=0.6)

ax.set_xticks(x_positions)
ax.set_xticklabels(list(schema_data.keys()), fontsize=9)
ax.set_xlabel("Schema (sample_size=1000 for all — see Limitations)")
ax.set_ylabel("Measured epsilon (mean +/- std)")
ax.set_title("Epsilon vs. Schema Complexity\n(LIMITATION: sample_size not varied)")
ax.grid(visible=True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Correlation Preservation

`correlation_matrix_delta` = Frobenius norm of (real correlation matrix - synthetic).
Lower delta = better preservation of inter-column relationships.

**Key observations:**
- Customers: all deltas = 0.0 (no meaningful numeric correlations)
- Orders: deltas ~1.26-1.39 (numeric columns customer_id, total_amount)
- No monotonic improvement with lower epsilon at this grid scale


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for rows, label, color, marker in [
    (customers_rows, "Customers (8 cols)", "#2563eb", "o"),
    (orders_rows, "Orders (5 cols)", "#16a34a", "s"),
]:
    completed = [r for r in rows if r["status"] == "COMPLETED"]
    data = [
        (r["actual_epsilon"], r["correlation_matrix_delta"])
        for r in completed
        if r.get("actual_epsilon") is not None and r.get("correlation_matrix_delta") is not None
    ]
    if data:
        xs, ys = zip(*data, strict=False)
        ax.scatter(list(xs), list(ys), label=label, color=color, marker=marker, s=80, zorder=3)

ax.set_xlabel("Measured epsilon")
ax.set_ylabel("Correlation Matrix Delta (Frobenius norm)")
ax.set_title(
    "Correlation Preservation vs. Epsilon\n"
    "(customers delta=0; orders delta>0 due to numeric columns)"
)
ax.legend(loc="upper left", fontsize=9)
ax.grid(visible=True, alpha=0.3)
plt.tight_layout()
plt.show()

## FK Integrity Verification

`fk_orphan_rate` measures what proportion of synthesised child rows reference
a parent ID that does not exist in the synthesised parent table.

For CSV-source runs (this benchmark), `fk_orphan_rate = null`. This is expected:
the benchmark harness synthesises each table independently. FK coherence is enforced
at the subsetting/masking pipeline level, which is outside the scope of this benchmark.

The table below shows all runs — no results are filtered or hidden.


In [ ]:
all_combined = [{**r, "_schema": "customers"} for r in customers_rows] + [
    {**r, "_schema": "orders"} for r in orders_rows
]

table_data = [
    [
        r["_schema"],
        f"sigma={r['noise_multiplier']:.1f}",
        str(r["epochs"]),
        r.get("status", "UNKNOWN"),
        str(r.get("fk_orphan_rate")) if r.get("fk_orphan_rate") is not None else "null",
    ]
    for r in all_combined
]
col_labels = ["Schema", "Noise Mult.", "Epochs", "Status", "FK Orphan Rate"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.axis("off")
tbl = ax.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.2, 1.4)

for row_idx, row in enumerate(table_data):
    if row[3] == "FAILED":
        for col_idx in range(len(col_labels)):
            tbl[row_idx + 1, col_idx].set_facecolor("#fecaca")

ax.set_title(
    "FK Integrity Verification\n"
    "(fk_orphan_rate=null for CSV-source runs — no FK enforcement at synthesis)",
    pad=10,
)
plt.tight_layout()
plt.show()

## Honest Limitations

### 1. Reduced Grid

The executed grid covers 6 cells x 2 datasets = 12 runs total. The originally
planned grid was substantially larger (135+ cells per dataset). Conclusions from
12 data points should be treated as preliminary indicators, not definitive benchmarks.

**Critical:** sample_size variation was dropped entirely. All runs used sample_size=1000.
Any claim about how epsilon scales with dataset size is unsupported by this benchmark.

### 2. CTGAN Architecture Constraints

CTGAN's GAN-based architecture has known limitations:
- **Datetime columns**: CTGAN encodes datetime as ordinal integers, producing very low
  chi2 p-values for datetime columns (e.g., created_at achieves p ~4.4e-103). This is
  a known CTGAN deficiency, not a DP-specific issue.
- **Convergence**: 50-100 epochs may be insufficient for CTGAN to converge.
  Statistical fidelity may improve substantially with 300-1000 epochs.
- **Small dataset**: 1000 rows is small for CTGAN. Larger datasets generally give
  better fidelity.

### 3. Budget Exhaustion at sigma=1.0, epochs=100 (Customers)

The customers run with sigma=1.0 and epochs=100 failed with BudgetExhaustionError.
This is correct behaviour — the DP accountant refused to proceed because the requested
epsilon exceeded the configured budget ceiling. This run is included in all tables
and charts as a red X / FAILED row.

### 4. What These Numbers Mean

These epsilon values (epsilon ~1.2-40) represent the theoretical privacy loss upper bound
under the Renyi Differential Privacy accountant with delta=1e-5.

- epsilon < 2: generally considered 'strong' DP for practical applications
- epsilon ~5-10: typical for well-configured SDG pipelines
- epsilon > 10: limited formal privacy guarantees

The low-epsilon runs (sigma=10.0) achieve epsilon ~1.2-3.0, which is practically
meaningful. However, statistical fidelity suggests the model is not converged —
low epsilon with poor fidelity is not a useful product.

### 5. What These Numbers Do NOT Mean

- They do NOT guarantee that no individual record can be reconstructed
- They do NOT account for post-processing attacks or composition with other DP mechanisms
- They do NOT reflect production configuration (different seeds, larger grids, GPU, more epochs)
- fk_orphan_rate=null does NOT mean FK integrity is guaranteed

### 6. cuDNN Non-Determinism

These benchmarks ran CPU-only (GPU not available). On GPU-equipped machines, cuDNN's
non-deterministic algorithms may produce different epsilon values even with the same seed.
Reproducibility on GPU requires torch.backends.cudnn.deterministic=True.


## Pre-Rendered Figures

The SVG figures are committed to `demos/figures/` and generated by
`demos/generate_figures.py`. To regenerate:

```bash
poetry run python demos/generate_figures.py
```

| Figure | File |
|--------|------|
| Epsilon vs. Noise Multiplier | `demos/figures/epsilon_vs_noise_multiplier.svg` |
| Epsilon vs. Statistical Fidelity | `demos/figures/epsilon_vs_statistical_fidelity.svg` |
| Epsilon vs. Schema Complexity | `demos/figures/epsilon_vs_schema_complexity.svg` |
| Correlation Preservation | `demos/figures/correlation_preservation.svg` |
| FK Integrity | `demos/figures/fk_integrity.svg` |
